# Project 3 — Explainable AI

**Goal:** Explain a Random Forest model with permutation feature importance. The breast-cancer benchmark is for learning only, not medical decisions.

Run the cells from top to bottom. Data is generated locally or loaded from scikit-learn; no CSV download is required.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, silhouette_score, mean_absolute_error, mean_squared_error,
    precision_score, recall_score
)
from sklearn.datasets import load_breast_cancer, make_classification, make_blobs
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
print("Setup complete")


## Project 3 — Explainable AI (Random Forest + Permutation Importance)
**Goal:** Train a model and explain which input features influence its predictions. This dataset is the built-in breast-cancer benchmark; use it only for learning, not for medical decisions.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

xai_model = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE
)
xai_model.fit(X_train, y_train)
pred = xai_model.predict(X_test)
proba = xai_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, target_names=data.target_names))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))

In [ ]:
# Model-agnostic explanation using permutation importance
perm = permutation_importance(
    xai_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring="roc_auc"
)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
display(importance.head(10).to_frame("mean_auc_decrease"))

importance.head(10).sort_values().plot(kind="barh", figsize=(8, 5))
plt.xlabel("Mean decrease in ROC-AUC after shuffling")
plt.title("Permutation feature importance")
plt.tight_layout()
plt.show()

In [ ]:
# Optional SHAP explanation (requires: pip install shap)
# Uncomment and run if SHAP is installed.
# import shap
# explainer = shap.TreeExplainer(xai_model)
# shap_values = explainer.shap_values(X_test.iloc[:100])
# shap.summary_plot(shap_values, X_test.iloc[:100], feature_names=X.columns)

## Possible extensions
- Install `shap` and enable the optional SHAP cell above.
- Compare global feature importance with explanations for individual predictions.